In [1]:
from tasks.autoencoder import AETask
import os
import torch

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [5]:
classifier.device

device(type='cuda', index=0)

In [2]:
sequence_to_classify = "Bart was building a deck. He went to the lumberyard to buy supplies. He bought seventeen boards. He couldn't fit them in his truck, so the lumberyard delivered. He appreciated their good customer service."

In [ ]:
labels = [
    'family',
    'friendship',
    'romance',
    'parenting',
    'childhood',
    'helping',
    'conflict',
    'deception',
    'forgiveness',
    'school',
    'studying',
    'teaching',
    'work',
    'unemployment',
    'achievement',
    'setback',
    'collaboration',
    'leadership',
    'competition',
    'shopping',
    'cooking',
    'cleaning',
    'housing',
    'commuting',
    'finances',
    'technology',
    'hobbies',
    'pets',
    'entertainment',
    'illness',
    'doctor',
    'exercise',
    'sports',
    'stress',
    'accident',
    'ae_task-improvement',
    'addiction',
    'safety',
    'recovery',
    'travel',
    'celebration',
    'surprise',
    'weather',
    'adventure',
    'grief',
    'crime',
    'aspiration',
    'change',
    'morality',
    'values'
]

In [15]:
out = classifier(sequence_to_classify, labels, multi_label=True)

In [19]:
torch.Tensor(out['scores']) > 0.9

tensor([ True,  True,  True,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])

In [ ]:
out['labels']

In [22]:
from tasks.autoencoder import AETask

In [2]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                '/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints',
                'jvktv590',
                "last.ckpt",
            ),
            strict=False,
        )
ae_task.setup()

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [24]:
dataloader = ae_task.val_dataloader()
batch = next(iter(dataloader))

In [25]:
# move batch to cuda and float16
batch = {k: v.cuda() if k != 'input_str' else v for k, v in batch.items()}

In [26]:
ae_task.half()
ae_task.eval()
with torch.no_grad():
    loss = 0.0

    input_ids_enc_clean = batch["input_ids_enc"]
    input_ids_enc_corrupted = ae_task.random_substitution(input_ids_enc_clean, p=0.3)

    z_clean = ae_task.encoder(
        input_ids_enc_clean, attention_mask=batch["attention_mask_enc"]
    )
    z_corrupted = ae_task.encoder(
        input_ids_enc_corrupted, attention_mask=batch["attention_mask_enc"]
    )

    if ae_task.encoder.cfg.variational:
        mean_clean, log_val_clean = z_clean
        KLD = -0.5 * torch.sum(
            1 + log_val_clean - mean_clean.pow(2) - log_val_clean.exp()
        )
        ae_task.log("val/KLD", KLD, on_epoch=True)
        loss += ae_task.cfg.kl_beta * KLD

        # Note: use mean for evaluation
        z_clean = mean_clean
        z_corrupted = z_corrupted[0]

    # Evaluate loss for clean inputs
    logits = ae_task.decoder(
        batch["input_ids_dec"], z_clean, attention_mask=batch["attention_mask_dec"]
    )
    targets = batch["input_ids_dec"].masked_fill(
        batch["attention_mask_dec"] == 0, -1
    )
    logits = logits[:, :-1].contiguous()
    targets = targets[:, 1:].contiguous()
    recon_loss = torch.nn.functional.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        ignore_index=-1,
    )
    ae_task.log("val/reconstruction_loss", recon_loss, on_epoch=True)
    loss += recon_loss

    # Decode from clean and corrupted latents and evaluate BLEU
    gen_clean, gen_corrupted = [
        ae_task.decoder.tokenizer.batch_decode(
            ae_task.decoder.generate_from(
                z, max_length=ae_task.cfg.max_generation_length
            ),
            skip_special_tokens=True,
        )
        for z in (z_clean, z_corrupted)
    ]

    bleu_clean = ae_task.bleu.compute(
        predictions=gen_clean, references=batch["input_str"]
    )["bleu"]
    bleu_corrupted = ae_task.bleu.compute(
        predictions=gen_corrupted, references=batch["input_str"]
    )["bleu"]

    ae_task.log("val/bleu_clean", bleu_clean, on_epoch=True)
    ae_task.log("val/bleu_corrupted", bleu_corrupted, on_epoch=True)

    # Interpolate between pairs of clean latents and evaluate perplexity
    group_indices = torch.randperm(input_ids_enc_clean.shape[0]).chunk(2)
    z_groups = [z_clean[indices] for indices in group_indices]
    z_interp = 0.5 * z_groups[0] + 0.5 * z_groups[1]
    gen_interp_ids = ae_task.decoder.generate_from(
        z_interp, max_length=ae_task.cfg.max_generation_length
    )
    gen_interp_str = ae_task.decoder.tokenizer.batch_decode(
        gen_interp_ids, skip_special_tokens=True
    )

    with ae_task.decoder.backbone.disable_adapter():
        # Disable LoRA layers for perplexity eval
        mask = (gen_interp_ids != ae_task.decoder.tokenizer.pad_token_id)
        labels = gen_interp_ids.masked_fill(~mask, -100)
        ppl_interp = torch.exp(
            ae_task.decoder.backbone(
                input_ids=gen_interp_ids,
                labels=labels,
                attention_mask=mask,
            ).loss
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/lightning/pytorch/core/module.py:449: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


In [28]:
ae_task.bleu.compute(
        predictions=gen_interp_str, references=batch["input_str"]
    )["bleu"]

ValueError: Mismatch in the number of predictions (64) and references (128)

In [27]:
gen_interp_str

['Kyle was moving to a new house. He needed help wearing his furniture. Sometimes Kyle used his other friends. He decided to help his friends with the furniture. Now Kyle can see things to compliment them.',
 'Sam invited Jill to her party. The morning of it, she put him the bag of candy. She was so excited to see everyone! No one bit into candy, though. Sam cried.',
 'Bill worked hard all day in order to study for the important test. When he had made sure his knowledge had reached this red convertible, he gave it a lap. He drove the test way, and played with it down at times. While he was taking the test, he got a blister on the way for it. He got it to pass with an A, and the test a blue!',
 'It was time for Jack to get his new furniture for his birthday. He was told the piano would be called, and arrive to noon. Jack waited, but no one had arrived. When they called the furniture store  they would arrive to see it. Just then Jack was moved by the man with the piano.',
 'The flag flew